Importamos las librerias de Spark

In [0]:
!pip install pyspark

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

Paso 1. Cargar y asignar nombres a las columnas

Creamos la sesion

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, to_date

spark = SparkSession.builder.appName("PanaderiaZerrillo").getOrCreate()

Cargamos el archivo de datos csv de la panaderia Zerrillo

In [0]:
df = spark.read.csv("/Volumes/workspace/bd_zerrillo/grupo_01/26058/RESULTADOS-PanaderiaZerrillo.csv",header=False,sep=";",inferSchema=True,)

Damos Nombres a lasa columnas

In [0]:
df = df.toDF(
    "id_venta",
    "fecha",
    "producto",
    "categoria",
    "cantidad",
    "precio",
    "metodo_pago"
)

Paso 2. Revisar la estructura de los datos

In [0]:
# Mostrar tipos de datos
df.printSchema()

# Mostrar estadísticas generales
df.describe().show()

# Contar registros
print("Total de registros:", df.count())

root
 |-- id_venta: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio: double (nullable = true)
 |-- metodo_pago: string (nullable = true)

+-------+-----------------+--------------------+------------------+------------------+------------------+-----------+
|summary|         id_venta|            producto|         categoria|          cantidad|            precio|metodo_pago|
+-------+-----------------+--------------------+------------------+------------------+------------------+-----------+
|  count|             1767|                1767|              1767|              1767|              1767|       1767|
|   mean|656.5563101301641|                NULL|              NULL|2.8007923033389925|2.9431239388794515|       NULL|
| stddev|383.9014877435355|                NULL|              NULL|2.4889198276975826|3.1231893015446053|       NULL|
|    mi

Paso 3. Eliminar espacios innecesarios

In [0]:
# Limpiar espacios en columnas de texto
df = df.withColumn("producto", trim(col("producto"))).withColumn("categoria", trim(col("categoria"))).withColumn("metodo_pago", trim(col("metodo_pago")))

df.show(5, truncate=False)

+--------+----------+---------------------+--------------------+--------+------+-----------+
|id_venta|fecha     |producto             |categoria           |cantidad|precio|metodo_pago|
+--------+----------+---------------------+--------------------+--------+------+-----------+
|1       |2026-09-01|Pan francés (par x 2)|Panes               |2       |1.0   |Yape       |
|2       |2026-09-01|Aceite Capri 1 L     |Despensa y abarrotes|1       |9.3   |Yape       |
|3       |2026-09-01|Pan francés (par x 2)|Panes               |4       |2.0   |Plin       |
|4       |2026-09-01|Pan francés (par x 2)|Panes               |2       |1.0   |Efectivo   |
|4       |2026-09-01|Queso fresco 100 g   |Embutidos y quesos  |1       |2.0   |Efectivo   |
+--------+----------+---------------------+--------------------+--------+------+-----------+
only showing top 5 rows


Paso 4. Identificar y tratar valores nulos

In [0]:
# Revisar valores nulos
df.select([col(c).isNull().cast("int").alias(c)
    for c in df.columns
]).groupBy().sum().show()

df = df.na.drop(
    subset=[
        "id_venta",
        "fecha",
        "producto",
        "categoria",
        "cantidad",
        "precio",
        "metodo_pago"
    ]
)

+-------------+----------+-------------+--------------+-------------+-----------+----------------+
|sum(id_venta)|sum(fecha)|sum(producto)|sum(categoria)|sum(cantidad)|sum(precio)|sum(metodo_pago)|
+-------------+----------+-------------+--------------+-------------+-----------+----------------+
|            0|         0|            0|             0|            0|          0|               0|
+-------------+----------+-------------+--------------+-------------+-----------+----------------+



Paso 5. Corregir tipos de datos

In [0]:
df = df.withColumn("id_venta", col("id_venta").cast("integer")).withColumn("fecha", to_date(col("fecha"), "yyyy-MM-dd")).withColumn("cantidad", col("cantidad").cast("integer")).withColumn("precio", col("precio").cast("decimal(12,2)"))

df.printSchema()

root
 |-- id_venta: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio: decimal(12,2) (nullable = true)
 |-- metodo_pago: string (nullable = true)



Paso 6. Filtrar registros incorrectos

In [0]:
df = df.filter(
    (col("id_venta") > 0) &
    (col("fecha").isNotNull()) &
    (col("producto") != "") &
    (col("categoria") != "") &
    (col("cantidad") > 0) &
    (col("precio") > 0) &
    (col("metodo_pago") != "")
)

df.show(10, truncate=False)

+--------+----------+----------------------+--------------------+--------+------+-----------+
|id_venta|fecha     |producto              |categoria           |cantidad|precio|metodo_pago|
+--------+----------+----------------------+--------------------+--------+------+-----------+
|1       |2026-09-01|Pan francés (par x 2) |Panes               |2       |1.00  |Yape       |
|2       |2026-09-01|Aceite Capri 1 L      |Despensa y abarrotes|1       |9.30  |Yape       |
|3       |2026-09-01|Pan francés (par x 2) |Panes               |4       |2.00  |Plin       |
|4       |2026-09-01|Pan francés (par x 2) |Panes               |2       |1.00  |Efectivo   |
|4       |2026-09-01|Queso fresco 100 g    |Embutidos y quesos  |1       |2.00  |Efectivo   |
|5       |2026-09-01|Pan ciabatta (par x 2)|Panes               |6       |3.00  |Yape       |
|6       |2026-09-01|Pan francés (par x 2) |Panes               |4       |2.00  |Efectivo   |
|7       |2026-09-01|Pan francés (par x 2) |Panes           

Paso 7. Eliminar registros duplicados

In [0]:
df = df.dropDuplicates()

print("Registros después de eliminar duplicados:", df.count())

Registros después de eliminar duplicados: 1767


Paso 8. Crear una nueva columna calculada

In [0]:
df_total = df = df.withColumn(
    "importe_total",
    col("precio")
)

In [0]:
from pyspark.sql.functions import sum as spark_sum

df.select(
    spark_sum("cantidad").alias("Total_Unidades"),
    spark_sum("importe_total").alias("Total_Ventas")
).show()


+--------------+------------+
|Total_Unidades|Total_Ventas|
+--------------+------------+
|          4949|     5200.50|
+--------------+------------+



Paso 9. Seleccionar columnas necesarias

In [0]:
# Seleccionar información para el análisis
df = df.select(
    "id_venta",
    "fecha",
    "producto",
    "categoria",
    "cantidad",
    "precio",
    "importe_total",
    "metodo_pago"
)

df.show(10, truncate=False)

+--------+----------+----------------------+--------------------+--------+------+-------------+-----------+
|id_venta|fecha     |producto              |categoria           |cantidad|precio|importe_total|metodo_pago|
+--------+----------+----------------------+--------------------+--------+------+-------------+-----------+
|2       |2026-09-01|Aceite Capri 1 L      |Despensa y abarrotes|1       |9.30  |9.30         |Yape       |
|10      |2026-09-01|Jamón especial 100 g  |Embutidos y quesos  |1       |2.00  |2.00         |Efectivo   |
|19      |2026-09-01|Pan ciabatta (par x 2)|Panes               |2       |1.00  |1.00         |Efectivo   |
|26      |2026-09-01|Pan francés (par x 2) |Panes               |2       |1.00  |1.00         |Yape       |
|53      |2026-09-01|Pan francés (par x 2) |Panes               |4       |2.00  |2.00         |Yape       |
|73      |2026-09-01|Pan francés (par x 2) |Panes               |6       |3.00  |3.00         |Efectivo   |
|84      |2026-09-01|Pan cia

Paso 10. Agrupar y analizar las ventas

Agrupar las ventas por categoría:

In [0]:
df_ventas_categoria=df.groupBy("categoria").sum("cantidad", "importe_total")
display(df_ventas_categoria)

categoria,sum(cantidad),sum(importe_total)
Despensa y abarrotes,96,464.50
Embutidos y quesos,66,363.40
Panes,4362,2744.50
Infusiones y café,33,33.00
Panetones y dulces,32,126.10
Lácteos y yogures,130,772.50
Galletas envasadas,58,95.60
Bebidas,59,237.70
Panes y preparados,24,48.00
Limpieza e higiene,26,160.40


Agrupar las ventas por método de pago:

In [0]:
df_ventas_pago=df.groupBy("metodo_pago").sum("importe_total")
display(df_ventas_pago)

metodo_pago,sum(importe_total)
Yape,1370.20
Efectivo,3442.80
Plin,387.50


Filtrar productos de la categoría Panes:

In [0]:
df_panes_c=df.filter(
    col("categoria") == "Panes"
).select(
    "producto",
    "cantidad",
    "precio",
    "importe_total"
)
display(df_panes_c)

producto,cantidad,precio,importe_total
Pan ciabatta (par x 2),2,1.00,1.00
Pan francés (par x 2),2,1.00,1.00
Pan francés (par x 2),4,2.00,2.00
Pan francés (par x 2),6,3.00,3.00
Pan ciabatta (par x 2),2,1.00,1.00
Pan ciabatta (par x 2),4,2.00,2.00
Pan ciabatta (par x 2),4,2.00,2.00
Pan francés (par x 2),4,2.00,2.00
Pan francés (par x 2),4,2.00,2.00
Pan francés (par x 2),4,2.00,2.00


Analizar las ventas por producto:

In [0]:
df.groupBy("producto") \
    .sum("cantidad", "importe_total") \
    .orderBy(col("sum(importe_total)").desc()) \
    .show(10, truncate=False)

+-----------------------------+-------------+------------------+
|producto                     |sum(cantidad)|sum(importe_total)|
+-----------------------------+-------------+------------------+
|Pan ciabatta (par x 2)       |2120         |1060.00           |
|Pan francés (par x 2)        |2113         |1056.50           |
|Pan integral (bolsa)         |46           |230.00            |
|Pan de yema largo (paquete)  |29           |145.00            |
|Yogur Gloria fresa 1 L       |13           |100.10            |
|Pan de yema redondo (paquete)|20           |100.00            |
|Pan hamburguesa (paquete)    |12           |94.80             |
|Yogur Milkito vainilla 1 L   |10           |82.00             |
|Hot dog Suiza paquete 5 und  |9            |81.90             |
|Panetón Gloria 900 g         |3            |72.60             |
+-----------------------------+-------------+------------------+
only showing top 10 rows


CREACION BASE DE DATOS

In [0]:
%sql

CREATE DATABASE IF NOT EXISTS bd_zerrillo;


In [0]:
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("bd_zerrillo.tb_ventas_zerrillo")

In [0]:

dfVentasGuardadas = spark.table('bd_zerrillo.tb_ventas_zerrillo')

display(dfVentasGuardadas)


id_venta,fecha,producto,categoria,cantidad,precio,importe_total,metodo_pago
2,2026-09-01,Aceite Capri 1 L,Despensa y abarrotes,1,9.30,9.30,Yape
10,2026-09-01,Jamón especial 100 g,Embutidos y quesos,1,2.00,2.00,Efectivo
19,2026-09-01,Pan ciabatta (par x 2),Panes,2,1.00,1.00,Efectivo
26,2026-09-01,Pan francés (par x 2),Panes,2,1.00,1.00,Yape
53,2026-09-01,Pan francés (par x 2),Panes,4,2.00,2.00,Yape
73,2026-09-01,Pan francés (par x 2),Panes,6,3.00,3.00,Efectivo
84,2026-09-01,Pan ciabatta (par x 2),Panes,2,1.00,1.00,Plin
98,2026-09-01,Pan ciabatta (par x 2),Panes,4,2.00,2.00,Yape
115,2026-09-01,Azúcar rubia por kilo,Despensa y abarrotes,1,4.00,4.00,Yape
145,2026-09-02,Pan ciabatta (par x 2),Panes,4,2.00,2.00,Efectivo


GRAFICOS

In [0]:
dfVentasCategoria = df.groupBy("categoria").sum("importe_total")
display(dfVentasCategoria)

categoria,sum(importe_total)
Despensa y abarrotes,464.50
Embutidos y quesos,363.40
Panes,2744.50
Infusiones y café,33.00
Panetones y dulces,126.10
Lácteos y yogures,772.50
Galletas envasadas,95.60
Bebidas,237.70
Panes y preparados,48.00
Limpieza e higiene,160.40


Databricks visualization. Run in Databricks to view.

In [0]:

dfProductosVendidos = df.groupBy("producto").sum("cantidad")

dfProductosVendidos = dfProductosVendidos.orderBy(
    col("sum(cantidad)").desc()
).limit(10)

display(dfProductosVendidos)


producto,sum(cantidad)
Pan ciabatta (par x 2),2120
Pan francés (par x 2),2113
Pan integral (bolsa),46
Pan de yema largo (paquete),29
Tamal tradicional,24
Pan de yema redondo (paquete),20
Empanaditas con ajonjolí (4 und),17
Empanaditas con grajeas (4 und),15
Pan caracol (par x 2),14
Yogur Gloria fresa 1 L,13


Databricks visualization. Run in Databricks to view.

In [0]:

dfVentasPago = df.groupBy("metodo_pago").sum("importe_total")

display(dfVentasPago)


metodo_pago,sum(importe_total)
Yape,1370.20
Efectivo,3442.80
Plin,387.50


Databricks visualization. Run in Databricks to view.

In [0]:

import time
from functools import reduce

# Dataset base
df_base = df

# Volúmenes de prueba
multiplicadores = [1, 2, 5, 10]

resultados = []

for factor in multiplicadores:

    # Generar volumen de prueba mediante UNION
    data_prueba = reduce(
        lambda a, b: a.union(b),
        [df_base] * factor
    )

    inicio = time.perf_counter()

    # Ejecutar procesamiento
    total_registros = data_prueba.count()

    fin = time.perf_counter()

    tiempo = fin - inicio

    resultados.append((
        factor,
        total_registros,
        round(tiempo, 3)
    ))

# Mostrar resultados
columnas = [
    "Multiplicador",
    "Total_Registros",
    "Tiempo_Segundos"
]

df_resultados = spark.createDataFrame(
    resultados, columnas
)

display(df_resultados)


Multiplicador,Total_Registros,Tiempo_Segundos
1,1767,0.591
2,3534,0.689
5,8835,0.97
10,17670,1.177


Databricks visualization. Run in Databricks to view.

In [0]:

from pyspark.sql.functions import col, sum as spark_sum

# Corregir el importe total
df_bi = df.withColumn(
    "importe_total",
    col("cantidad") * col("precio")
)

# 1. Ventas por categoría
ventas_categoria = df_bi.groupBy("categoria").agg(
    spark_sum("cantidad").alias("Total_Unidades"),
    spark_sum("importe_total").alias("Total_Ventas")
).orderBy(col("Total_Ventas").desc())

display(ventas_categoria)

# 2. Ventas por método de pago
ventas_pago = df_bi.groupBy("metodo_pago").agg(
    spark_sum("importe_total").alias("Total_Ventas")
).orderBy(col("Total_Ventas").desc())

display(ventas_pago)

# 3. Productos más vendidos
productos_vendidos = df_bi.groupBy("producto").agg(
    spark_sum("cantidad").alias("Total_Unidades"),
    spark_sum("importe_total").alias("Total_Ventas")
).orderBy(col("Total_Unidades").desc()).limit(10)

display(productos_vendidos)


categoria,Total_Unidades,Total_Ventas
Panes,4362,12812.30
Lácteos y yogures,130,1042.70
Despensa y abarrotes,96,560.70
Embutidos y quesos,66,394.20
Panetones y dulces,32,281.90
Bebidas,59,237.70
Limpieza e higiene,26,160.40
Galletas envasadas,58,135.00
Cereales,26,117.80
Panes y preparados,24,80.00


Databricks visualization. Run in Databricks to view.

metodo_pago,Total_Ventas
Efectivo,10751.20
Yape,3910.80
Plin,1266.70


Databricks visualization. Run in Databricks to view.

producto,Total_Unidades,Total_Ventas
Pan ciabatta (par x 2),2120,6259.00
Pan francés (par x 2),2113,5664.50
Pan integral (bolsa),46,400.00
Pan de yema largo (paquete),29,175.00
Tamal tradicional,24,80.00
Pan de yema redondo (paquete),20,130.00
Empanaditas con ajonjolí (4 und),17,29.00
Empanaditas con grajeas (4 und),15,25.00
Pan caracol (par x 2),14,25.00
Yogur Gloria fresa 1 L,13,177.10


Databricks visualization. Run in Databricks to view.

In [0]:

from pyspark.sql.functions import col, sum as spark_sum

# Calcular importe total
df_evolucion = df.withColumn(
    "importe_total",
    col("cantidad") * col("precio")
)

# Agrupar las ventas por fecha
ventas_diarias = df_evolucion.groupBy("fecha").agg(
    spark_sum("importe_total").alias("Total_Ventas"),
    spark_sum("cantidad").alias("Total_Unidades")
).orderBy("fecha")

# Mostrar resultados
display(ventas_diarias)


fecha,Total_Ventas,Total_Unidades
2026-09-01,1568.30,530
2026-09-02,1700.80,526
2026-09-03,1888.00,516
2026-09-04,1831.40,551
2026-09-05,1648.90,535
2026-09-06,1404.10,421
2026-09-07,1801.60,528
2026-09-08,1669.50,482
2026-09-09,1364.50,448
2026-09-10,1051.60,412


Databricks visualization. Run in Databricks to view.